# 02. book_text Ablation

3개 Cross-encoder × 5가지 book_text variant × 21시나리오 = **315번** 실험

- **입력**: `experiments/rerank/data/gold_candidate_pool.json`
- **출력**: `experiments/rerank/data/results/ablation/{variant}_{model}.json` × 15개
- **선정 기준**: MRR@10 평균 기준 모델별 최적 variant

In [11]:
import importlib.util
import json
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'NanumGothic'

# CWD에서 위로 올라가며 evaluation/ 폴더가 있는 프로젝트 루트를 탐색
def find_root(marker: str = 'evaluation') -> Path:
    p = Path.cwd().resolve()
    while p != p.parent:
        if (p / marker).is_dir():
            return p
        p = p.parent
    raise FileNotFoundError(f"프로젝트 루트 못 찾음 ('{marker}/' 없음)")

ROOT       = find_root('evaluation')          # .../somin (= librAIan)
RERANK_DIR = ROOT / 'experiments' / 'rerank'

sys.path.insert(0, str(RERANK_DIR / 'src'))

from book_text_variants import VARIANTS, VARIANT_DESC
from cross_encoder_reranker import CrossEncoderReranker, MODELS

# metrics.py를 파일 경로로 직접 로드
_metrics_path = ROOT / 'evaluation' / 'metrics' / 'metrics.py'
_spec = importlib.util.spec_from_file_location('metrics', _metrics_path)
_mod  = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)
mrr_at_k  = _mod.mrr_at_k
ndcg_at_k = _mod.ndcg_at_k
hit_at_k  = _mod.hit_at_k

DATA_DIR   = RERANK_DIR / 'data'
RESULT_DIR = DATA_DIR / 'results' / 'ablation'
RESULT_DIR.mkdir(parents=True, exist_ok=True)

RELEVANT_THRESHOLD = 2
DEVICE = 'cuda'  # GPU 없으면 'cpu'

print(f'ROOT: {ROOT}')
print(f'CWD:  {Path.cwd()}')

ROOT: /home/ubuntu/work/somin
CWD:  /home/ubuntu/work/somin/backend


In [12]:
# 현재 실행 위치 확인
print(f"Current working directory: {Path.cwd()}")

Current working directory: /home/ubuntu/work/somin/backend


## 1. Gold Candidate Pool 로드

In [13]:
with open(DATA_DIR / 'gold_candidate_pool.json') as f:
    gold_pool = json.load(f)

print(f'시나리오 수: {len(gold_pool)}')
print(f'총 후보 수: {sum(s["n_candidates"] for s in gold_pool)}')

시나리오 수: 21
총 후보 수: 1678


## 2. 실험 헬퍼

In [14]:
def run_scenario(reranker, scenario: dict, text_fn) -> dict:
    query = scenario['semantic_query']
    candidates = scenario['candidates']

    reranked = reranker.rerank(query, candidates, text_fn)
    reranked_isbns = [r['book']['isbn'] for r in reranked]

    relevant_isbns = [
        c['isbn'] for c in candidates if c['final_grade'] >= RELEVANT_THRESHOLD
    ]
    relevance_scores = {c['isbn']: c['final_grade'] for c in candidates}

    return {
        'scenario_id': scenario['scenario_id'],
        'n_candidates': len(candidates),
        'n_relevant': len(relevant_isbns),
        'mrr@10':  mrr_at_k(relevant_isbns, reranked_isbns, k=10),
        'ndcg@10': ndcg_at_k(relevance_scores, reranked_isbns, k=10),
        'hit@10':  hit_at_k(relevant_isbns, reranked_isbns, k=10),
        'mrr@5':   mrr_at_k(relevant_isbns, reranked_isbns, k=5),
        'ndcg@5':  ndcg_at_k(relevance_scores, reranked_isbns, k=5),
        'hit@5':   hit_at_k(relevant_isbns, reranked_isbns, k=5),
        'reranked_isbns': reranked_isbns[:20],
    }


def avg_metric(results: list[dict], metric: str) -> float:
    vals = [r[metric] for r in results if r['n_relevant'] > 0]
    return sum(vals) / len(vals) if vals else 0.0

## 3. Ablation 실험 실행

> 모델 로드에 시간이 걸릴 수 있다. 모델 하나씩 실행 후 결과를 저장한다.

In [15]:
# 실행할 모델/variant 조합 선택 (전체 실행: None)
TARGET_MODELS   = None  # 예: ['BGE'] 로 특정 모델만
TARGET_VARIANTS = None  # 예: ['A', 'B']

model_names   = list(MODELS.keys())   if TARGET_MODELS   is None else TARGET_MODELS
variant_names = list(VARIANTS.keys()) if TARGET_VARIANTS is None else TARGET_VARIANTS

all_summary = []  # {model, variant, avg_mrr@10, avg_ndcg@10, avg_hit@10}

for model_name in model_names:
    print(f'\n── 모델 로드: {model_name} ({MODELS[model_name]}) ──')
    reranker = CrossEncoderReranker(MODELS[model_name], device=DEVICE)

    for variant_id in variant_names:
        text_fn = VARIANTS[variant_id]
        print(f'  variant {variant_id}: {VARIANT_DESC[variant_id]}')

        scenario_results = []
        for scenario in gold_pool:
            res = run_scenario(reranker, scenario, text_fn)
            scenario_results.append(res)

        summary = {
            'model': model_name,
            'variant': variant_id,
            'avg_mrr@10':  avg_metric(scenario_results, 'mrr@10'),
            'avg_ndcg@10': avg_metric(scenario_results, 'ndcg@10'),
            'avg_hit@10':  avg_metric(scenario_results, 'hit@10'),
            'avg_mrr@5':   avg_metric(scenario_results, 'mrr@5'),
            'scenarios': scenario_results,
        }
        all_summary.append(summary)

        out_path = RESULT_DIR / f'{variant_id}_{model_name}.json'
        with open(out_path, 'w', encoding='utf-8') as f:
            json.dump(summary, f, ensure_ascii=False, indent=2)

        print(f'    MRR@10={summary["avg_mrr@10"]:.4f}  NDCG@10={summary["avg_ndcg@10"]:.4f}  Hit@10={summary["avg_hit@10"]:.4f}')

    del reranker  # 다음 모델을 위해 메모리 해제

print('\n실험 완료.')


── 모델 로드: BGE (BAAI/bge-reranker-v2-m3) ──


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 6194.84it/s]


  variant A: 도서명 + 카테고리 + 책소개 (baseline)
    MRR@10=0.6288  NDCG@10=0.5796  Hit@10=1.0000
  variant B: A + 목차
    MRR@10=0.6933  NDCG@10=0.5652  Hit@10=1.0000
  variant C: A + 저자 + 출판사
    MRR@10=0.6639  NDCG@10=0.5653  Hit@10=1.0000
  variant D: A + 검색순위 + 검색점수
    MRR@10=0.6512  NDCG@10=0.5746  Hit@10=1.0000
  variant E: 전체 필드 (현재 운영)
    MRR@10=0.6802  NDCG@10=0.5550  Hit@10=0.9524

── 모델 로드: GTE (Alibaba-NLP/gte-multilingual-reranker-base) ──


ValueError: Alibaba-NLP/new-impl You can inspect the repository content at https://hf.co/Alibaba-NLP/gte-multilingual-reranker-base.
Please pass the argument `trust_remote_code=True` to allow custom code to be run.

## 4. 결과 분석: 모델별 최적 Variant 선정

In [16]:
# 저장된 결과 파일에서 로드 (재실행 가능)
loaded = []
for path in sorted(RESULT_DIR.glob('*.json')):
    with open(path) as f:
        d = json.load(f)
    loaded.append({'model': d['model'], 'variant': d['variant'],
                   'avg_mrr@10': d['avg_mrr@10'], 'avg_ndcg@10': d['avg_ndcg@10'],
                   'avg_hit@10': d['avg_hit@10']})

df = pd.DataFrame(loaded)

# 피벗: 모델 × variant → MRR@10
pivot = df.pivot(index='variant', columns='model', values='avg_mrr@10')
print('=== MRR@10 (variant × model) ===')
print(pivot.to_string(float_format='{:.4f}'.format))

# 모델별 최적 variant
best_variant = pivot.idxmax()
print('\n=== 모델별 최적 variant (MRR@10 기준) ===')
for model, var in best_variant.items():
    print(f'  {model}: Variant {var}  ({VARIANT_DESC[var]})')

=== MRR@10 (variant × model) ===
model      BGE
variant       
A       0.6288
B       0.6933
C       0.6639
D       0.6512
E       0.6802

=== 모델별 최적 variant (MRR@10 기준) ===
  BGE: Variant B  (A + 목차)


In [ ]:
# 시각화
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
metrics = ['avg_mrr@10', 'avg_ndcg@10', 'avg_hit@10']
titles  = ['MRR@10', 'NDCG@10', 'Hit@10']

for ax, metric, title in zip(axes, metrics, titles):
    pivot_m = df.pivot(index='variant', columns='model', values=metric)
    pivot_m.plot(kind='bar', ax=ax, legend=(metric == 'avg_mrr@10'))
    ax.set_title(title)
    ax.set_xlabel('Variant')
    ax.tick_params(axis='x', rotation=0)

plt.suptitle('book_text Variant Ablation', fontsize=13)
plt.tight_layout()
plt.savefig(DATA_DIR / 'results' / 'ablation_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 3단계로 넘길 최적 variant 저장
best = {model: var for model, var in best_variant.items()}
with open(DATA_DIR / 'results' / 'best_variant_per_model.json', 'w') as f:
    json.dump(best, f, indent=2)
print('저장:', best)